# 06 · Capstone: A Regional Study

**Goal: run the full pipeline end to end, then extend it into a project.**

You now have every piece: discovery, querying, chunked extraction, analysis, and
visualization. This capstone chains them into one workflow over a geographic
region. It adds two things you need for real fieldwork:

- **A map of the nodes.** Coordinates are more legible on a map than in a table,
  and coloring the markers by value shows at a glance which nodes agree and which
  one disagrees.
- **A data quality filter.** Real sensor feeds contain obvious errors: a flat 0,
  a large negative like -200, a stuck high value. These sit right next to the
  plausible readings and will wreck any statistic if you leave them in. You will
  build two complementary filters to remove them.

A **region** is a bounding box: a range of latitudes and a range of longitudes.
Any node whose coordinates fall inside the box is "in" the region. For reference,
the Chicago box is roughly latitude 41.64 to 42.02 and longitude -87.94 to
-87.52.

This notebook runs both ways. With a network it studies real nodes end to end.
Offline it studies three synthetic Chicago nodes through the same code path, with
deliberate flaws baked in (one miscalibrated node, one node with injected error
spikes) so the map and the filters have something to show. It prints which mode it
is in at every step.

In [ ]:
# Offline-safe synthetic data generator.
#
# Produces records in the exact schema sage_data_client.query returns, so every
# SageTools helper works on it unchanged. Use it when there is no route to the
# Sage Data API. Timestamps are fixed to mid 2024 and values are generated, not
# measured, so this data is never mistaken for real readings.

import numpy as np
import pandas as pd


def makeSyntheticBeehive(vsn="W07C", days=14, stepMinutes=5, seed=7,
                         sensors=("bme680", "bme280")):
    """Build a Beehive-schema temperature DataFrame (Celsius) with a diurnal
    cycle, seasonal drift, per-sensor bias, and two outage gaps."""
    rng = np.random.default_rng(seed)
    periods = int(days * 24 * 60 / stepMinutes)
    times = pd.date_range("2024-06-01", periods=periods, freq=f"{stepMinutes}min", tz="UTC")

    hourOfDay = times.hour + times.minute / 60.0
    diurnal = 8.0 * np.sin((hourOfDay - 9.0) / 24.0 * 2 * np.pi)
    dayIndex = (times - times[0]).days
    seasonalDrift = 0.15 * dayIndex
    baseline = 21.0

    frames = []
    for offsetIdx, sensor in enumerate(sensors):
        sensorBias = 0.6 * offsetIdx
        noise = rng.normal(0, 0.4, size=periods)
        celsius = baseline + diurnal + seasonalDrift + sensorBias + noise
        frames.append(pd.DataFrame({
            "timestamp": times,
            "name": "env.temperature",
            "value": celsius,
            "meta.vsn": vsn,
            "meta.sensor": sensor,
            "meta.host": f"000048b02d{15 + offsetIdx}a.ws-nxcore",
        }))

    combined = pd.concat(frames, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    for gapStart, gapHours in [(3, 6), (9, 14)]:
        lo = times[0] + pd.Timedelta(days=gapStart)
        hi = lo + pd.Timedelta(hours=gapHours)
        combined = combined[~combined["timestamp"].between(lo, hi)].reset_index(drop=True)
    return combined

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab also shows the underlying sage_data_client call, so
# the helper is never a hard dependency. To install it, see notebook 00.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

In [ ]:
import sage_data_client
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display


def ensureFolium():
    """
    Import folium and branca, installing folium into this kernel if it is missing.

    Uses sys.executable so the install targets the exact environment backing this
    kernel, and a normal non-editable install so the import works without a
    restart. Returns (foliumModule, colormapModule), or (None, None) if the
    install could not complete (for example, no network).
    """
    try:
        import folium
        import branca.colormap as colormap
        return folium, colormap
    except ImportError:
        import sys, subprocess, importlib
        print("folium not found, installing it into this kernel...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "folium"],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            lastLine = (result.stderr.strip().splitlines() or ["unknown error"])[-1]
            print("folium install failed:", lastLine)
            print("maps will be skipped; the rest of the notebook runs normally.")
            return None, None
        importlib.invalidate_caches()
        import folium
        import branca.colormap as colormap
        print("folium installed")
        return folium, colormap


folium, cm = ensureFolium()
haveFolium = folium is not None

## 1. Define the region

We use a small dictionary of bounding boxes so the notebook is self contained. If
you installed SageTools, `RegionManager` gives you the same thing plus lookup of
new cities by name.

In [ ]:
regionBoxes = {
    "chicago": {"lat": [41.6445, 42.0230], "lon": [-87.9401, -87.5241]},
    "usa": {"lat": [24.0, 49.0], "lon": [-125.0, -66.0]},
}

targetRegion = "chicago"
box = regionBoxes[targetRegion]
print(f"studying {targetRegion}: lat {box['lat']}, lon {box['lon']}")

if haveSageTools:
    from sage.common.regions import RegionManager
    manager = RegionManager()
    print("known SageTools regions:", manager.listRegions())

## 2. Find the nodes in the region

Three steps, each a small query you have seen before.

1. **Which nodes are alive?** Query the `sys.uptime` heartbeat across the network
   over a recent window and collect the distinct call signs.
2. **Where is each node?** A temperature record does not always carry
   coordinates, so we get them from the `sys.gps.lat` and `sys.gps.lon` metrics,
   taking each node's most recent fix.
3. **Which sit inside the box?** A range check on the coordinates.

Offline, all three steps use a fixed set of three synthetic nodes placed inside
Chicago.

In [ ]:
# The offline synthetic nodes, all inside the Chicago box. Two carry deliberate
# flaws so the map and filters have something to reveal:
#   - miscalibratedNodes: a node with a large warm bias, plausible but well above
#     its peers, so it stands out on the value-colored map.
#   - errorProneNodes: a node that gets error spikes injected, so the data-quality
#     filters have something to remove.
syntheticCoords = {
    "W06C": (41.87, -87.65),
    "W020": (41.79, -87.60),
    "W039": (42.01, -87.70),
}
miscalibratedNodes = {"W039": 22.0}   # warm bias in Celsius, a teaching device
errorProneNodes = {"W020"}            # gets injected error spikes, a teaching device


def findLiveNodes(minutes=60):
    """
    Discover live nodes from sys.uptime heartbeats.

    Returns (liveVsns, isLive). isLive is True for real Beehive data and False
    when it falls back to the synthetic node set.
    """
    try:
        beatDf = sage_data_client.query(start=f"-{minutes}m", filter={"name": "sys.uptime"})
        if not beatDf.empty:
            liveVsns = sorted(beatDf["meta.vsn"].dropna().astype(str).unique().tolist())
            print(f"using live Beehive data: {len(liveVsns)} nodes sent a heartbeat "
                  f"in the last {minutes} minutes")
            return liveVsns, True
    except Exception as err:
        print("live query unavailable:", err)

    print("using synthetic fallback: three placed Chicago nodes")
    return sorted(syntheticCoords.keys()), False


liveVsns, isLive = findLiveNodes(minutes=60)

In [ ]:
def getNodeLocations(vsns, isLive, gpsWindow="-2d"):
    """
    Build a {vsn: (lat, lon)} map for the given nodes.

    Live: the most recent sys.gps.lat and sys.gps.lon fix per node.
    Offline: the fixed synthetic coordinates.
    """
    if not isLive:
        return {vsn: syntheticCoords[vsn] for vsn in vsns if vsn in syntheticCoords}

    locations = {}
    try:
        gpsLatDf = sage_data_client.query(start=gpsWindow, filter={"name": "sys.gps.lat"})
        gpsLonDf = sage_data_client.query(start=gpsWindow, filter={"name": "sys.gps.lon"})
    except Exception as err:
        print("GPS lookup failed:", err)
        return locations

    if gpsLatDf.empty or gpsLonDf.empty:
        print("no GPS records returned, cannot place nodes")
        return locations

    latLatest = gpsLatDf.sort_values("timestamp").groupby("meta.vsn")["value"].last()
    lonLatest = gpsLonDf.sort_values("timestamp").groupby("meta.vsn")["value"].last()
    for vsn in vsns:
        if vsn in latLatest.index and vsn in lonLatest.index:
            locations[vsn] = (float(latLatest[vsn]), float(lonLatest[vsn]))

    print(f"located {len(locations)} of {len(vsns)} live nodes via sys.gps")
    return locations


nodeLocations = getNodeLocations(liveVsns, isLive)

In [ ]:
def nodesInBox(locations, box):
    """Return the sorted VSNs whose coordinates fall inside the region box."""
    latLo, latHi = box["lat"]
    lonLo, lonHi = box["lon"]
    return sorted(
        vsn for vsn, (lat, lon) in locations.items()
        if latLo <= lat <= latHi and lonLo <= lon <= lonHi
    )


regionNodes = nodesInBox(nodeLocations, box)
print(f"nodes inside {targetRegion}: {regionNodes}")

## 3. Put the nodes on a map

A table of coordinates is hard to read. The map below drops a marker at each
located node. This first map is plain: it answers "where are they?" A later map
will color the same markers by value.

The map is interactive when you have internet, because the background tiles load
from OpenStreetMap in your browser. The markers themselves are computed locally,
so the map builds fine offline even if the tiles do not appear.

In [ ]:
# Satellite basemap source, used for the value-colored node maps below. The
# supplement notebook (supplement_folium_maps.ipynb) explains map styles in full.
esriSatelliteTiles = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
                      "World_Imagery/MapServer/tile/{z}/{y}/{x}")
esriAttribution = "Tiles (c) Esri, Maxar, Earthstar Geographics"


def makeNodeMap(locations, nodeValues=None, caption="node mean temperature (F)",
                tiles="OpenStreetMap", attr=None):
    """
    Build a folium map of node locations.

    If nodeValues (a {vsn: number} map) is given, markers are colored on a scale
    and any node far from the group is outlined in red. Otherwise markers are
    plain. Pass tiles=esriSatelliteTiles with attr=esriAttribution for satellite
    imagery. Returns a folium.Map, which renders inline in Jupyter.
    """
    if not locations:
        print("no located nodes to map")
        return None

    lats = [lat for lat, _ in locations.values()]
    lons = [lon for _, lon in locations.values()]
    center = [sum(lats) / len(lats), sum(lons) / len(lons)]
    fmap = folium.Map(location=center, zoom_start=10, tiles=tiles, attr=attr)

    scale = None
    groupMedian = groupMad = None
    if nodeValues:
        values = [v for v in nodeValues.values() if v is not None]
        if values:
            scale = cm.LinearColormap(["blue", "green", "yellow", "red"],
                                      vmin=min(values), vmax=max(values))
            scale.caption = caption
            fmap.add_child(scale)
            valueSeries = pd.Series(values)
            groupMedian = valueSeries.median()
            groupMad = (valueSeries - groupMedian).abs().median()

    for vsn, (lat, lon) in locations.items():
        value = nodeValues.get(vsn) if nodeValues else None
        if value is None or scale is None:
            fillColor = "#3186cc"
            outline = "#3186cc"
            radius = 6
            popup = f"{vsn}"
        else:
            fillColor = scale(value)
            isOutlier = (groupMad is not None and groupMad > 0
                         and abs(value - groupMedian) > 3 * 1.4826 * groupMad)
            outline = "red" if isOutlier else fillColor
            radius = 10 if isOutlier else 7
            flag = "  (disagrees with the group)" if isOutlier else ""
            popup = f"{vsn}: {value:.1f} F{flag}"
        folium.CircleMarker(
            location=[lat, lon], radius=radius, color=outline, weight=3,
            fill=True, fill_color=fillColor, fill_opacity=0.85, popup=popup,
        ).add_to(fmap)

    return fmap


if haveFolium:
    locationMap = makeNodeMap(nodeLocations)
    display(locationMap)
else:
    print("folium unavailable, skipping the map.")

## 4. Clean the data before you trust it

Real sensor feeds are noisy in a way that ordinary statistics cannot survive. A
disconnected probe may report a flat 0, a firmware fault may emit a code like
-200, a sun struck sensor may spike high. A single -200 mixed into readings that
otherwise sit near 70 will drag the mean down by a large amount and make the
minimum meaningless. So you filter before you summarize.

We use two filters, and they catch different things.

**Filter one, a physical plausibility band.** Temperature has hard physical
limits. Nothing reads -200 F, so we drop anything outside a band you set from
domain knowledge, which catches gross errors like -200 and 999. Choosing the band
matters: the Sage `env.temperature` sensor sits near the node electronics and
reads hot, often up to 140 F or more rather than shaded air temperature, so this
notebook uses a ceiling of 160 F. If you set the ceiling too low, the filter
silently trims the tops off your hottest nodes and their lines flatten along the
ceiling in the plot. That flattening is the tell: if you see it, raise maxValid.
Set the band to your own metric and sensor.

**Filter two, a per-node robust outlier check.** Some errors sit inside the
physical band, like a flat 0 among readings near 70. Rather than hardcode a band,
we measure how far each reading is from that node's own median, scaled by the
median absolute deviation (MAD). The MAD is itself resistant to the errors it is
trying to find, and it adapts to whatever a node's normal level is, so it works
whether a node reads near 70 or near 120.

In [ ]:
def dropImplausible(frame, valueCol="valueF", minValid=-40.0, maxValid=160.0):
    """
    Remove readings outside a physically plausible band.

    The band is a hard physical limit for the metric, not a statistical one. For
    the Sage temperature feed, which runs hot, -40 to 160 keeps real peaks while
    removing gross errors like -200 and 999. Set it to match your metric.

    Returns (cleanedFrame, removedCount).
    """
    numeric = pd.to_numeric(frame[valueCol], errors="coerce")
    keep = numeric.between(minValid, maxValid)
    return frame.loc[keep].copy(), int((~keep).sum())


def dropOutliersMad(frame, valueCol="valueF", threshold=3.5):
    """
    Remove readings far from the node's own median, using the median absolute
    deviation (MAD). Robust to the errors it removes, and adaptive to whatever the
    node's normal level is, so no fixed band is assumed.

    A reading is dropped when its distance from the median exceeds threshold times
    the scaled MAD. The default 3.5 is the standard modified z-score cutoff
    (Iglewicz and Hoaglin), aggressive enough to catch a flat 0 sitting among
    readings near 70, while keeping real daily temperature extremes.
    Returns (cleanedFrame, removedCount).
    """
    numeric = pd.to_numeric(frame[valueCol], errors="coerce")
    median = numeric.median()
    mad = (numeric - median).abs().median()
    if mad == 0:
        return frame.copy(), 0                 # no spread, nothing to flag
    scaledMad = 1.4826 * mad                    # scale so MAD estimates the std
    keep = (numeric - median).abs() <= threshold * scaledMad
    return frame.loc[keep].copy(), int((~keep).sum())


def injectErrors(frame, valueCol="valueF", n=20, seed=0):
    """Insert obvious sensor-error values into a copy of a frame, for teaching."""
    if len(frame) == 0:
        return frame.copy()
    rng = np.random.default_rng(seed)
    dirty = frame.copy().reset_index(drop=True)
    errorValues = np.array([0.0, -50.0, -200.0, 999.0])
    idx = rng.choice(dirty.index, size=min(n, len(dirty)), replace=False)
    dirty.loc[idx, valueCol] = rng.choice(errorValues, size=len(idx))
    return dirty


print("filters defined: dropImplausible, dropOutliersMad, injectErrors")

### Watch the filters work on one node

Before wiring the filters into the pipeline, run them on a single node so you can
see exactly what they remove. Offline, the chosen node has error spikes injected,
so the counts are non zero and the recovered clean range is obvious.

In [ ]:
def fetchNodeTemperature(vsn, isLive, start="-1d"):
    """
    Return a tidy temperature frame for one node, live or synthetic.

    Offline, deliberate flaws are added: a warm bias for miscalibrated nodes and
    injected error spikes for error-prone nodes, both teaching devices. The frame
    has a parsed timestamp and a Fahrenheit column, or is empty when a live node
    reported nothing in the window.
    """
    if isLive:
        nodeDf = sage_data_client.query(
            start=start, filter={"name": "env.temperature", "vsn": vsn})
        if nodeDf.empty:
            return nodeDf
        # A node can carry more than one temperature sensor, and they can read very
        # differently: a shaded-air sensor near 80 F and a board sensor near 120 F.
        # Mixing them makes a node's mean meaningless and shows up as two bands, so
        # keep the single most common sensor. Pick a specific sensor here if you
        # know which one you want.
        if "meta.sensor" in nodeDf.columns and nodeDf["meta.sensor"].nunique() > 1:
            topSensor = nodeDf["meta.sensor"].value_counts().index[0]
            nodeDf = nodeDf[nodeDf["meta.sensor"] == topSensor]
        nodeDf["timestamp"] = pd.to_datetime(nodeDf["timestamp"])
        nodeDf["valueF"] = nodeDf["value"] * 9 / 5 + 32
        return nodeDf.sort_values("timestamp").reset_index(drop=True)

    # Synthetic branch, with deterministic per-node character.
    nodeSeed = sum(ord(ch) for ch in vsn)
    nodeDf = makeSyntheticBeehive(vsn=vsn, days=7, sensors=("bme680",), seed=nodeSeed)
    nodeDf["timestamp"] = pd.to_datetime(nodeDf["timestamp"])
    siteOffsetC = (nodeSeed % 5) - 2                        # small site to site spread
    biasC = miscalibratedNodes.get(vsn, 0.0)               # demo miscalibration
    nodeDf["value"] = nodeDf["value"] + siteOffsetC + biasC
    nodeDf["valueF"] = nodeDf["value"] * 9 / 5 + 32
    if vsn in errorProneNodes:
        nodeDf = injectErrors(nodeDf, valueCol="valueF", n=20, seed=nodeSeed)
    return nodeDf.sort_values("timestamp").reset_index(drop=True)


demoVsn = (regionNodes or sorted(nodeLocations.keys()) or [None])[0]
rawDf = None
cleanDemoDf = None
removedDemoDf = None

if demoVsn is None:
    print("No node available to demonstrate filtering.")
else:
    # Start from the node's real (or synthetic) single-sensor readings, then inject
    # a handful of obvious errors so the filters have something visible to remove.
    # The injection is a teaching device applied in both live and offline mode, so
    # this demonstration always makes its point. The real analysis in section 5
    # filters the data without injecting anything.
    cleanReadings = fetchNodeTemperature(demoVsn, isLive, start="-1d")
    rawDf = injectErrors(cleanReadings, valueCol="valueF", n=25, seed=1)
    print(f"node {demoVsn}: {len(rawDf)} readings, 25 of them overwritten with "
          f"demo errors")
    print(f"raw range with errors: "
          f"{rawDf['valueF'].min():.1f} to {rawDf['valueF'].max():.1f} F")

    step1Df, removedBand = dropImplausible(rawDf, "valueF", minValid=-40.0, maxValid=160.0)
    print(f"physical band [-40, 160] F removed {removedBand} readings")

    cleanDemoDf, removedMad = dropOutliersMad(step1Df, "valueF", threshold=3.5)
    print(f"per-node MAD filter removed {removedMad} more readings")

    removedDemoDf = rawDf.loc[~rawDf.index.isin(cleanDemoDf.index)]
    print(f"clean: {len(cleanDemoDf)} readings kept, {len(removedDemoDf)} removed in total, "
          f"range {cleanDemoDf['valueF'].min():.1f} to {cleanDemoDf['valueF'].max():.1f} F")

### See it: the same node before and after filtering

Counts are informative but a picture is clearer. We take the node's real readings,
inject 25 obvious errors so there is something to catch, then filter. The left
panel shows every reading with the injected errors marked as red crosses; the
spikes stretch the axis so far that the real daily signal is squeezed into a thin
band. The right panel shows only the survivors, where the daily pattern is legible
again. Your own analysis would not inject errors; this is only to make the filters'
effect visible.

In [ ]:
if rawDf is None:
    print("No demo node available, skipping the before and after plot.")
else:
    fig, (axBefore, axAfter) = plt.subplots(1, 2, figsize=(14, 5))

    # Before: every raw reading, with the removed points marked in red.
    axBefore.plot(rawDf["timestamp"], rawDf["valueF"], ".", markersize=3,
                  color="tab:blue", label="raw reading")
    if removedDemoDf is not None and not removedDemoDf.empty:
        axBefore.plot(removedDemoDf["timestamp"], removedDemoDf["valueF"], "x",
                      markersize=8, color="red", label="removed as error")
    axBefore.set_title(f"{demoVsn} before filtering: {len(rawDf)} readings")
    axBefore.set_xlabel("Time")
    axBefore.set_ylabel("Temperature (F)")
    axBefore.legend(loc="upper right")
    axBefore.grid(True, linestyle="--", alpha=0.5)
    axBefore.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    # After: only the surviving readings, on their own scale.
    axAfter.plot(cleanDemoDf["timestamp"], cleanDemoDf["valueF"], ".", markersize=3,
                 color="tab:green")
    axAfter.set_title(f"{demoVsn} after filtering: {len(cleanDemoDf)} readings")
    axAfter.set_xlabel("Time")
    axAfter.set_ylabel("Temperature (F)")
    axAfter.grid(True, linestyle="--", alpha=0.5)
    axAfter.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    for ax in (axBefore, axAfter):
        for tickLabel in ax.get_xticklabels():
            tickLabel.set_rotation(45)

    plt.tight_layout()
    plt.show()

## 5. Collect, clean, analyze, and visualize every node

Now run the full pipeline over every node to study: fetch, clean with both
filters, summarize, and plot a smoothed line. Cleaning happens before any
statistic is computed, so the means and extremes in the table are trustworthy.

Two robustness details, which together fix the empty legend warning: if no node
fell inside the region, we fall back to the geolocated live nodes so the figure is
never empty, and we only draw the legend when at least one line was plotted.

In [ ]:
def fetchCleanTemperature(vsn, isLive, start="-1d",
                          minValid=-40.0, maxValid=160.0, madThreshold=3.5):
    """Fetch a node's temperature and remove implausible and outlier readings.
    Returns (cleanFrame, removedCount)."""
    raw = fetchNodeTemperature(vsn, isLive, start=start)
    if raw.empty:
        return raw, 0
    step1, n1 = dropImplausible(raw, "valueF", minValid, maxValid)
    step2, n2 = dropOutliersMad(step1, "valueF", madThreshold)
    return step2, n1 + n2


# Choose which nodes to study. Prefer nodes inside the region. If none fell there,
# fall back to the geolocated live nodes so the pipeline still produces a figure.
nodesToStudy = list(regionNodes)
studyLabel = f"{targetRegion.title()} region"

if not nodesToStudy:
    fallbackNodes = sorted(nodeLocations.keys())[:5]
    if fallbackNodes:
        print(f"No live nodes fell inside {targetRegion}. Showing "
              f"{len(fallbackNodes)} geolocated live nodes instead.")
        nodesToStudy = fallbackNodes
        studyLabel = "live nodes, no region match"
    else:
        print("No geolocated nodes available. Widen the region box, increase the")
        print("findLiveNodes window, or force the synthetic demo (isLive = False).")

fig, ax = plt.subplots(figsize=(12, 6))
summaryRows = []
nodeMeans = {}
keptCounts = {}
removedCounts = {}
plottedAny = False

for vsn in nodesToStudy:
    cleanDf, removed = fetchCleanTemperature(vsn, isLive, start="-1d")
    if cleanDf.empty:
        print(f"{vsn}: no usable temperature in the window, skipping")
        continue

    smoothed = cleanDf.set_index("timestamp")["valueF"].rolling("1h", min_periods=1).mean()
    ax.plot(smoothed.index, smoothed.values, linewidth=1.2, label=vsn)
    plottedAny = True

    meanF = round(cleanDf["valueF"].mean(), 1)
    nodeMeans[vsn] = meanF
    keptCounts[vsn] = len(cleanDf)
    removedCounts[vsn] = removed
    summaryRows.append({
        "vsn": vsn,
        "kept": len(cleanDf),
        "removed": removed,
        "meanF": meanF,
        "minF": round(cleanDf["valueF"].min(), 1),
        "maxF": round(cleanDf["valueF"].max(), 1),
    })

if plottedAny:
    ax.set_xlabel("Time")
    ax.set_ylabel("Temperature (F)")
    ax.set_title(f"{studyLabel}, cleaned, 1 hour average per node")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    for tickLabel in ax.get_xticklabels():
        tickLabel.set_rotation(45)

    # One legend entry per node becomes unreadable past a handful of nodes, and an
    # inline legend covers the data. For a few nodes place it inline; for many,
    # move it outside to the right in columns with a small font, and reserve space
    # with the tight_layout rect so it is not clipped.
    nodeCount = len(summaryRows)
    if nodeCount <= 8:
        ax.legend(loc="best", fontsize="small")
        fig.tight_layout()
    else:
        legendCols = 2 if nodeCount <= 30 else 3
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5),
                  ncol=legendCols, fontsize="x-small", frameon=False,
                  title=f"{nodeCount} nodes")
        fig.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()

    print(pd.DataFrame(summaryRows).to_string(index=False))
else:
    print("Nothing to plot. See the messages above for how to proceed.")

## 6. Map the nodes by value: who agrees, who does not

You cannot draw a whole time series on a map, but you can color each node by a
single representative number, here its cleaned mean temperature. Nodes with
similar means get similar colors, so a cluster of agreement is obvious. A node
whose mean is far from the group median is outlined in red and enlarged, so a
miscalibrated or oddly sited node jumps out.

Offline, the miscalibrated node carries a warm bias, so it should appear in a
warmer color and, if it is far enough from its peers, with a red outline. This map
uses satellite imagery so you can see each node's surroundings; the plain location
map above uses the street basemap for orientation. The supplement notebook covers
switching map styles.

In [ ]:
if haveFolium and nodeMeans:
    mappable = {vsn: nodeLocations[vsn] for vsn in nodeMeans if vsn in nodeLocations}
    valueMap = makeNodeMap(mappable, nodeValues=nodeMeans,
                           caption="node mean temperature (F)",
                           tiles=esriSatelliteTiles, attr=esriAttribution)
    display(valueMap)

    groupMedian = pd.Series(list(nodeMeans.values())).median()
    maxDistance = max(abs(m - groupMedian) for m in nodeMeans.values())
    print("group median of node means:", round(groupMedian, 1), "F")
    for vsn, meanF in sorted(nodeMeans.items(), key=lambda kv: kv[1]):
        marker = "  <- furthest from the group" if abs(meanF - groupMedian) == maxDistance else ""
        print(f"  {vsn}: {meanF} F{marker}")
elif not haveFolium:
    print("folium unavailable, skipping the value-colored map.")
else:
    print("No node means available to map.")

## 7. All nodes, then the nodes you can trust

The map above shows every node. In practice you also want the subset you can trust
for a given study. We label a node **suspect** when its cleaned mean sits far from
the group (a modified z-score outlier of the per-node means), or when the filters
had to remove more than a small fraction of its readings. Everything else is
**good**.

A node can have messy raw data and still be good: if the filters cleaned it up and
its mean agrees with its neighbors, its data is usable. The node that stays suspect
is the one whose whole level disagrees, which points to miscalibration or unusual
siting rather than a few bad readings.

In [ ]:
def classifyNodes(nodeMeans, keptCounts, removedCounts,
                  meanZThreshold=3.5, maxRemovedFraction=0.05):
    """
    Label each node "good" or "suspect".

    A node is suspect when its cleaned mean is a group outlier by the modified
    z-score (its distance from the median of the means exceeds meanZThreshold
    times the scaled MAD of the means), or when more than maxRemovedFraction of its
    readings were removed as errors. Otherwise it is good.

    Returns {vsn: {"status": str, "reason": str}}.
    """
    means = pd.Series(nodeMeans)
    groupMedian = means.median()
    mad = (means - groupMedian).abs().median()
    scaledMad = 1.4826 * mad if mad > 0 else 0.0

    quality = {}
    for vsn, meanF in nodeMeans.items():
        reasons = []
        if scaledMad > 0 and abs(meanF - groupMedian) > meanZThreshold * scaledMad:
            reasons.append("mean disagrees with the group")
        kept = keptCounts.get(vsn, 0)
        removed = removedCounts.get(vsn, 0)
        total = kept + removed
        if total > 0 and removed / total > maxRemovedFraction:
            reasons.append(f"{removed / total:.0%} of readings removed as errors")
        status = "suspect" if reasons else "good"
        reason = "; ".join(reasons) if reasons else "agrees with the group, low error rate"
        quality[vsn] = {"status": status, "reason": reason}
    return quality


if nodeMeans:
    nodeQuality = classifyNodes(nodeMeans, keptCounts, removedCounts)

    print("all nodes:")
    for vsn in sorted(nodeQuality):
        entry = nodeQuality[vsn]
        print(f"  {vsn}: {entry['status']:>7}  ({entry['reason']})")

    goodNodes = sorted(vsn for vsn, entry in nodeQuality.items()
                       if entry["status"] == "good")
    print("\ngood nodes only:", goodNodes)

    if haveFolium and goodNodes:
        goodLocations = {vsn: nodeLocations[vsn] for vsn in goodNodes
                         if vsn in nodeLocations}
        goodMeans = {vsn: nodeMeans[vsn] for vsn in goodNodes}
        goodMap = makeNodeMap(goodLocations, nodeValues=goodMeans,
                              caption="good node mean temperature (F)",
                              tiles=esriSatelliteTiles, attr=esriAttribution)
        display(goodMap)
    elif not haveFolium:
        print("folium unavailable, skipping the good-nodes map.")
else:
    print("No node results to classify.")

## 8. Report the finding, fact separate from interpretation

Fill in the sentences below from your own run.

- **Fact.** The study covered the nodes in the summary table. After cleaning, the
  filters removed the reading counts shown, and the mean temperatures were as
  listed, with one node furthest from the group median.
- **Interpretation, a hypothesis to test.** A node whose cleaned mean sits far
  from its neighbors could be genuinely warmer (siting in direct sun, a rooftop
  versus a shaded pole), or it could be miscalibrated. The map and the removed
  counts narrow this down but do not settle it. State what further check would
  distinguish a real microclimate from a sensor problem.

## 9. Project prompts

Pick one and take it as far as you can. Each is a small research question.

1. **Urban heat island.** Compare downtown nodes to nodes farther from the core
   over a hot week. Is the city center measurably warmer overnight? Quantify the
   overnight difference and state your confidence.
2. **Data reliability.** Across every node in a region, estimate uptime from
   `sys.uptime` coverage, and separately, the fraction of temperature readings your
   filters remove. Which nodes are cleanest? Present a ranked table and one honest
   caveat about how you defined each measure.
3. **Multi metric story.** Combine temperature with `env.relative_humidity` to
   estimate a comfort index over a week. Where and when was it least comfortable?
4. **Seasonality.** Using chunked extraction from notebook `03`, pull three months
   for one node and characterize how its daily temperature range shifts across the
   season.

### What a strong submission shows

- A clear question stated up front.
- Reproducible cells: anyone can rerun and reach your numbers.
- Data cleaned before it is summarized, with the removed counts reported.
- Facts and interpretations kept in separate sentences.
- At least one figure or map that makes the finding legible.
- A short limitations paragraph naming what your data cannot tell you.